# Multi-Dimensional Data

tsam_xarray handles multi-dimensional DataArrays through two mechanisms:

- **`cluster_dim`** — dimensions clustered together (shared clustering)
- **Auto-slicing** — remaining dimensions get independent clusterings

This notebook covers stacking, slicing, and weights.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30)
print(f"Dims: {list(da.dims)}")
print(f"Shape: {dict(da.sizes)}")

## Multiple cluster dims

Pass `cluster_dim=["variable", "region"]` to cluster all variable-region
combinations together. They are stacked internally and unstacked in the results.

In [ ]:
da_single = da.sel(scenario="low")

result = tsam_xarray.aggregate(
    da_single,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
)
print("Result dims:", result.cluster_representatives.dims)
result.cluster_representatives.to_dataframe("value").head(10)

In [ ]:
result.cluster_representatives.sel(variable="solar").plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Cluster representatives (solar, by region)",
)

## Auto-slicing

Any dimension not in `time_dim` or `cluster_dim` is automatically sliced —
one independent aggregation per coordinate, with results concatenated into
coherent multi-dimensional arrays.

Here, `scenario` is auto-sliced. Each scenario gets its own clustering.
Cluster weights, accuracy metrics, and cluster representatives all have the `scenario`
dimension — no manual looping or concatenation needed.

Without tsam_xarray, you'd have to:
```python
# Manual approach (what tsam_xarray replaces)
results = {}
for scenario in da.scenario.values:
    da_slice = da.sel(scenario=scenario)
    df = ...  # flatten to DataFrame
    results[scenario] = tsam.aggregate(df, n_clusters=4)
# Then manually concat cluster_weights, accuracy, cluster_representatives...
```

In [ ]:
result_sliced = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
)
print("Result dims:", result_sliced.cluster_representatives.dims)
result_sliced.cluster_weights.to_dataframe("weight")

In [ ]:
result_sliced.accuracy.rmse.to_dataframe("RMSE")

## Multiple slice dims

Only `variable` as `cluster_dim` — both `region` and `scenario` are auto-sliced.
One aggregation per (region, scenario) combination.

In [ ]:
result_multi = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
)
print("Result dims:", result_multi.cluster_representatives.dims)
result_multi.cluster_representatives.sel(
    variable="solar",
    scenario="low",
).plotly.line(
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Cluster representatives (solar, low, per region)",
)

## Weights

Use a dict to weight certain coordinates higher during clustering.
For multiple `cluster_dim`, use a dict-of-dicts keyed by dimension name.

In [ ]:
# Weight solar 2x — broadcasts across region (missing entries default to 1.0)
result_w = tsam_xarray.aggregate(
    da_single,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
    weights={"variable": {"solar": 2.0}},
)
result_w.accuracy.rmse.to_dataframe("RMSE")

Weights can span multiple dimensions — they multiply across dims:

In [ ]:
# Weight solar in north: solar=3.0 * north=2.0 = 6.0
result_w2 = tsam_xarray.aggregate(
    da_single,
    time_dim="time",
    cluster_dim=["variable", "region"],
    n_clusters=4,
    weights={"variable": {"solar": 3.0}, "region": {"north": 2.0}},
)
result_w2.accuracy.rmse.to_dataframe("RMSE")